In [ ]:
!pip install -U diffusers transformers accelerate -q

In [1]:
import torch, gc, requests
import matplotlib.pyplot as plt
from PIL import Image
from diffusers import StableDiffusionXLPipeline

print(f"GPU: {torch.cuda.get_device_name(0)}")

pipe = StableDiffusionXLPipeline.from_pretrained(
    "playgroundai/playground-v2.5-1024px-aesthetic",
    torch_dtype=torch.float16,
    use_safetensors=True,
)
pipe = pipe.to("cuda")

print("✅ الموديل جاهز")

GPU: Tesla T4


/usr/local/lib/python3.12/dist-packages/diffusers/utils/deprecation_utils.py:23: FutureWarning: `torch_dtype` is deprecated and will be removed in version 1.0.0. Please use `dtype` instead.
  deprecate("torch_dtype", "1.0.0", _TORCH_DTYPE_DEPRECATION_MESSAGE)


model_index.json:   0%|          | 0.00/684 [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 18 files:   0%|          | 0/18 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/517 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

✅ الموديل جاهز


In [ ]:
def brief_to_image_prompt(b):
    return f"{b['logo_direction']}, professional corporate branding, flat vector design, Adobe Illustrator style, geometric, minimalist icon, clean background, no text, no cartoon, no mascot"

briefs = requests.get("https://raw.githubusercontent.com/maram-elaian/brandora/main/data/test_briefs.json").json()[:5]

generated_files = []
for brief in briefs:
    print(f"⏳ {brief['id']}...")
    image = pipe(
        brief_to_image_prompt(brief),
        num_inference_steps=25,
        guidance_scale=3.0,
        generator=torch.Generator("cuda").manual_seed(42),
    ).images[0]

    filename = f"Playground_v2_{brief['id']}.png"
    image.save(filename)
    generated_files.append(filename)
    print(f"✅ {brief['id']}")

    del image
    gc.collect()
    torch.cuda.empty_cache()

print("✅ خلصت")